# Prepare 2601_XPCS scan 263
Load the raw detector frames, save the average holograms, and generate the preview PNGs that the user paints red on for the manual mask workflow.

In [ ]:
from pathlib import Path
import sys
import h5py
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/source_manifest.json").is_file())
sys.path.insert(0, str(ROOT))

from library import scan_io as inputs
from library.real_space_constraints import largest_support_component
from library.mask_store import (
    load_red_mask_png,
    mask_png_paths,
    save_mask_reference_png,
)

RAW_ROOT = Path("/media/riccardo/EXT_16TB1/DATA/P04/2601_XPCS/raw")
SCANS = [263]
DARK_SCANS = [262]
MASK_ROOT = ROOT / "data/2601_XPCS/masks"
AVERAGE_ROOT = ROOT / "data/2601_XPCS/averaged"
PREVIEW_ROOT = ROOT / "data/2601_XPCS/previews"
CACHE = ROOT / "data/2601_XPCS/prepared.h5"

ALIGN_HOLOGRAMS = True
ALIGNMENT_REFERENCE = 34
ALIGNMENT_MAX_SHIFT = 10.0
ALIGNMENT_UPSAMPLE_FACTOR = 20
ALIGNMENT_ROI = (940, 1060, 900, 980)
CROP = 410  # Geometry is applied once by reconstruction.
BINNING = 1
SATURATION = 65535


## 1. Inspect acquisition and save preview PNGs

In [ ]:
records = inputs.manifest(RAW_ROOT, SCANS)
print("States", len(records), "frames", sorted(set(r["frames"] for r in records)),
      "energy eV", sorted(set(r["energy_ev"] for r in records)))

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([r["current_A"] for r in records], "o-")
ax.set(xlabel="Acquisition index", ylabel="Magnet current (A)", title="Acquisition sequence; do not sort by current")
plt.show()

# Keep this notebook focused on the manual mask workflow: save the reference image,
# let the user paint the red mask, and then load the ready-made painted PNGs.
if not records:
    raise ValueError("No records found for the selected scan list.")

record = records[0]
raw, saturated, _ = inputs.frame_mean(record["path"], SATURATION)
mask_reference_png, mask_painted_png = mask_png_paths(MASK_ROOT, "mask_pixel")
support_reference_png, support_painted_png = mask_png_paths(MASK_ROOT, "supportmask")

if not mask_painted_png.exists() or not support_painted_png.exists():
    save_mask_reference_png(raw, mask_reference_png, log_scale=True, colormap="viridis")
    save_mask_reference_png(raw, support_reference_png, log_scale=True, colormap="viridis")
    raise FileNotFoundError(
        "Manual painted masks are missing. Paint the red mask on the two reference images "
        f"and save them as {mask_painted_png.name} and {support_painted_png.name}."
    )

mask = load_red_mask_png(mask_painted_png, expected_shape=raw.shape)
support = load_red_mask_png(support_painted_png, expected_shape=raw.shape)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[0].set_title("Mean raw frame")
axes[1].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[1].contour(mask, levels=[0.5], colors="r", linewidths=0.8)
axes[1].set_title("Detector mask")
axes[2].imshow(np.log1p(np.nan_to_num(raw)), cmap="viridis")
axes[2].contour(support, levels=[0.5], colors="r", linewidths=0.8)
axes[2].set_title("Support mask")
for ax in axes:
    ax.set_axis_off()
plt.show()


## 2. Save averaged holograms for manual mask painting

In [ ]:
dark_files = [path for scan in DARK_SCANS
              for path in sorted((RAW_ROOT / f"{scan:05d}" / 'marana').glob('*.nx'))]
config = inputs.prepare_cache(
    records, CACHE, dark_paths=dark_files, supportmask=support, mask_pixel=mask,
    crop=CROP, binning=BINNING, saturation=SATURATION,
    align_holograms=ALIGN_HOLOGRAMS, alignment_reference=ALIGNMENT_REFERENCE,
    alignment_max_shift=ALIGNMENT_MAX_SHIFT, alignment_upsample_factor=ALIGNMENT_UPSAMPLE_FACTOR,
    alignment_roi=ALIGNMENT_ROI)
# The largest painted aperture is the magnetic material; the others are vacuum references.
with h5py.File(CACHE, 'a') as handle:
    handle.create_dataset('material_mask', data=largest_support_component(support), compression='gzip')

AVERAGE_ROOT.mkdir(parents=True, exist_ok=True)
PREVIEW_ROOT.mkdir(parents=True, exist_ok=True)
with h5py.File(CACHE) as handle:
    for i, record in enumerate(records):
        stem = f"scan_{record['scan']:05d}_point_{record['point']:04d}_avg"
        image = handle['holograms'][i]
        with h5py.File(AVERAGE_ROOT / (stem+'.h5'), 'w') as average:
            average.create_dataset('hologram', data=image, compression='gzip')
            average.create_dataset('mask_pixel', data=handle['mask_pixel'][i], compression='gzip')
            average.attrs['normalization'] = config['normalization']
        save_mask_reference_png(image, PREVIEW_ROOT / (stem+'.png'), log_scale=True)
print(CACHE)
